# Dallas TL Attribution Run

This notebook previews the next-token prediction for:

```text
Fact: the capital of the state containing Dallas is
```

Then it runs the TransformerLens-backed attribution path through `BiologyAttributionRunner`, which imports `biology_server_t_lens` for the replacement-model forward and attribution pass. It saves the resulting attribution graph JSON plus an optional compact `.pt` summary.

## 1. Repo And Dependency Setup

Run this in a GPU Colab runtime. A high-memory A100 is the intended target for the default `BATCH_SIZE=128` / `MAX_FEATURE_NODES=300` settings.

In [2]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/rowan-dauria/llm-biology.git"
BRANCH = "bio-serv-transformer-lens"
DEFAULT_COLAB_REPO = Path("/content/llm-biology")


def find_existing_repo() -> Path | None:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "biology_server").is_dir() and (
            candidate / "biology_server_t_lens"
        ).is_dir():
            return candidate
    return None


repo_dir = find_existing_repo()
if repo_dir is None:
    if not (DEFAULT_COLAB_REPO / "biology_server").exists():
        subprocess.check_call(
            ["git", "clone", "--branch", BRANCH, REPO_URL, str(DEFAULT_COLAB_REPO)]
        )
    repo_dir = DEFAULT_COLAB_REPO

os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

local_circuit_tracer = repo_dir.parent / "circuit-tracer"
if local_circuit_tracer.exists() and str(local_circuit_tracer) not in sys.path:
    sys.path.insert(0, str(local_circuit_tracer))

print(f"Using repo: {repo_dir}")
print(f"Python: {sys.executable}")

Using repo: /content/llm-biology
Python: /usr/bin/python3


In [3]:
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
INSTALL_DEPS = IN_COLAB

if INSTALL_DEPS:
    packages = [
        "accelerate",
        "circuit-tracer",
        "einops",
        "fsspec",
        "huggingface-hub<1.0",
        "safetensors",
        "sentencepiece",
        "transformer-lens>=2.16.0",
        "transformers>=4.56.0,<=4.57.3",
        "tqdm",
        "zstandard",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
else:
    print("Skipping dependency install outside Colab. Set INSTALL_DEPS=True to force it.")

In [4]:
# Keep this notebook compatible with both circuit-tracer factory names.
import importlib
import sys

slt_module = importlib.import_module("circuit_tracer.transcoder.single_layer_transcoder")
if not hasattr(slt_module, "load_transcoder"):
    if not hasattr(slt_module, "load_relu_transcoder"):
        raise ImportError(
            "circuit_tracer.transcoder.single_layer_transcoder has neither "
            "load_transcoder nor load_relu_transcoder"
        )
    slt_module.load_transcoder = slt_module.load_relu_transcoder
    print("Added load_transcoder alias for this notebook runtime.")
else:
    print("circuit-tracer already exposes load_transcoder.")

# Drop any half-imported modules left by a failed earlier run in this kernel.
for module_name in list(sys.modules):
    if module_name == "biology_server" or module_name.startswith("biology_server."):
        del sys.modules[module_name]

Added load_transcoder alias for this notebook runtime.


In [5]:
# Optional: use a Colab secret named HF_TOKEN if you need authenticated HF downloads.
try:
    from google.colab import userdata
    from huggingface_hub import login

    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("Logged in to Hugging Face with HF_TOKEN.")
    else:
        print("No HF_TOKEN Colab secret found; continuing unauthenticated.")
except Exception as exc:
    print(f"HF login skipped: {exc}")

HF login skipped: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.


In [6]:
import torch

print(f"torch={torch.__version__}")
print(f"cuda_available={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu={torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"gpu_memory_gb={props.total_memory / 1024**3:.1f}")

torch=2.10.0+cu128
cuda_available=True
gpu=NVIDIA A100-SXM4-40GB
gpu_memory_gb=39.5


## 2. Configure The Run

`USE_CHAT_TEMPLATE=False` is intentional for this factual completion prompt. The preview target token is passed into graph generation so the attribution run targets exactly the token shown in the preview.

In [7]:
import inspect
from pathlib import Path

import biology_server.attribution as attribution_module
import biology_server_t_lens
from biology_server.attribution import DEFAULT_LAYERS, BiologyAttributionRunner

assert "biology_server_t_lens" in inspect.getsource(BiologyAttributionRunner._ensure_loaded)
assert "biology_server_t_lens" in inspect.getsource(BiologyAttributionRunner.generate_graph)
print(f"Using TL backend package: {biology_server_t_lens.__file__}")

PROMPT = "Fact: the capital of the state containing Dallas is"
SLUG = "dallas-state-capital-tl"
USE_CHAT_TEMPLATE = False

LAYERS = DEFAULT_LAYERS
MAX_FEATURE_NODES = 300
BATCH_SIZE = 128

OUTPUT_DIR = Path("data/colab_attribution_graphs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_PT_PATH = OUTPUT_DIR / f"{SLUG}.pt"

# Feature-example sidecars need data/feature_topk/150k-pile. They are not needed
# for the raw graph artifact, so skip them by default in Colab.
SKIP_FEATURE_EXAMPLES = True
if SKIP_FEATURE_EXAMPLES:

    def _skip_feature_examples(**_kwargs):
        print("[INFO] skipping feature-example sidecars")
        return {}

    attribution_module.build_feature_examples = _skip_feature_examples

runner = BiologyAttributionRunner(
    layers=LAYERS,
    graph_file_dir=OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    max_feature_nodes=MAX_FEATURE_NODES,
)

print(f"prompt={PROMPT!r}")
print(f"slug={SLUG}")
print(f"layers={LAYERS}")
print(f"output_dir={OUTPUT_DIR.resolve()}")

Using TL backend package: /content/llm-biology/biology_server_t_lens/__init__.py
prompt='Fact: the capital of the state containing Dallas is'
slug=dallas-state-capital-tl
layers=[2, 12, 24, 33]
output_dir=/content/llm-biology/data/colab_attribution_graphs


## 3. Preview The Logit Prediction

In [8]:
preview = runner.preview(
    PROMPT,
    slug=SLUG,
    use_chat_template=USE_CHAT_TEMPLATE,
)

print("Prompt tokens:")
for idx, token in enumerate(preview.prompt_tokens):
    print(f"  {idx:02d}: {token!r}")

print("\nTarget token:")
print(
    f"  id={preview.target_token_id} "
    f"text={preview.target_token_str!r} "
    f"prob={preview.target_token_prob:.6f}"
)

print("\nTop preview tokens:")
for item in preview.top_tokens:
    print(f"  id={item.token_id:<8} p={item.prob:.6f} text={item.token!r}")

[INFO] device=cuda dtype=torch.bfloat16 layers=[2, 12, 24, 33]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[DONE]  Loading preview model (27.6s)
[INFO] prompt tokens (10): ['Fact', ':', ' the', ' capital', ' of', ' the', ' state', ' containing', ' Dallas', ' is']
[START] Preview inference forward pass
[INFO] preview target token id=19260 (' Austin') p=0.9364
[DONE]  Preview inference forward pass (2.4s)
Prompt tokens:
  00: 'Fact'
  01: ':'
  02: ' the'
  03: ' capital'
  04: ' of'
  05: ' the'
  06: ' state'
  07: ' containing'
  08: ' Dallas'
  09: ' is'

Target token:
  id=19260 text=' Austin' prob=0.936377

Top preview tokens:
  id=19260    p=0.936377 text=' Austin'
  id=11002    p=0.022021 text=' Fort'
  id=220      p=0.006716 text=' '
  id=18542    p=0.004616 text=' Dallas'
  id=279      p=0.004616 text=' the'
  id=8257     p=0.004336 text=' Texas'
  id=7407     p=0.002048 text=' located'
  id=2598     p=0.001698 text=' called'
  id=55210    p=0.001322 text=' Irving'
  id=6515     p=0.001242 text=' Washington'


## 4. Run TransformerLens Attribution And Save The Graph

In [9]:
result = runner.generate_graph(
    PROMPT,
    slug=SLUG,
    target_token_id=preview.target_token_id,
    save_pt=str(SAVE_PT_PATH),
    use_chat_template=USE_CHAT_TEMPLATE,
)

print("Saved attribution graph:")
print(f"  graph_json={result.graph_path}")
print(f"  compact_pt={result.pt_path}")
print(f"  target={result.target_token_str!r} p={result.target_token_prob:.6f}")
print(f"  feature_nodes={len(result.selected_features)}")
print(f"  links={len(result.links)}")
print(f"  logit_targets={len(result.logit_targets)}")

[INFO] device=cuda dtype=torch.bfloat16 layers=[2, 12, 24, 33]


[START] Loading model


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen3-4B into HookedTransformer
[DONE]  Loading model (23.3s)
[START] Resolving transcoder paths


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

layer_33.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

layer_2.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

layer_24.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

[DONE]  Resolving transcoder paths (21.6s)
[START] Loading transcoder layer 2
  d_model=2560  d_transcoder=163840  skip=False
[DONE]  Loading transcoder layer 2 (0.5s)
[START] Loading transcoder layer 12
  d_model=2560  d_transcoder=163840  skip=False
[DONE]  Loading transcoder layer 12 (0.5s)
[START] Loading transcoder layer 24
  d_model=2560  d_transcoder=163840  skip=False
[DONE]  Loading transcoder layer 24 (0.5s)
[START] Loading transcoder layer 33
  d_model=2560  d_transcoder=163840  skip=False
[DONE]  Loading transcoder layer 33 (0.5s)
[INFO] loaded 0 feature labels
[INFO] prompt tokens (10): ['Fact', ':', ' the', ' capital', ' of', ' the', ' state', ' containing', ' Dallas', ' is']
[START] TL replacement forward pass
[INFO] primary logit id=19260 (' Austin') p=0.0000 logit=-2.891; logit nodes=1; active features=1107571
[DONE]  TL replacement forward pass (66.0s)
[START] TL iterative top-K attribution
[DONE]  TL iterative top-K attribution (0.6s)


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.81 GiB. GPU 0 has a total capacity of 39.49 GiB of which 917.44 MiB is free. Including non-PyTorch memory, this process has 38.59 GiB memory in use. Of the allocated memory 37.43 GiB is allocated by PyTorch, and 665.12 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 5. Inspect And Download Saved Artifacts

In [ ]:
import json

graph_path = Path(result.graph_path)
with graph_path.open(encoding="utf-8") as handle:
    graph = json.load(handle)

print(json.dumps(graph["metadata"], indent=2))
print(f"nodes={len(graph['nodes'])}")
print(f"links={len(graph['links'])}")
print(f"json_size_mb={graph_path.stat().st_size / 1024**2:.2f}")
if result.pt_path is not None:
    pt_path = Path(result.pt_path)
    print(f"pt_size_mb={pt_path.stat().st_size / 1024**2:.2f}")

In [ ]:
# Optional Colab download. The files are already saved under OUTPUT_DIR.
DOWNLOAD_FROM_COLAB = False

if DOWNLOAD_FROM_COLAB:
    from google.colab import files

    files.download(str(result.graph_path))
    if result.pt_path is not None:
        files.download(str(result.pt_path))